# Coolify Deployment

Coolify is a self-hosted PaaS. It wraps Docker, manages Traefik routing, and handles GitHub deployments.
This notebook covers the full deployment workflow: connecting a repo, setting up services, and troubleshooting.

## Concepts

| Term | What it means |
|------|---------------|
| **Server** | The VPS Coolify manages |
| **Project** | A logical group (e.g., mad-house, orinadus, personal) |
| **Environment** | Production, staging, etc. inside a project |
| **Application** | A GitHub repo being built and deployed |
| **Service** | A pre-built Docker image (postgres, redis, n8n) |
| **Resource** | Any app or service in Coolify |

Traefik is deployed automatically by Coolify and handles all routing and SSL.

## Adding a New Application

1. In Coolify: Project > Environment > Add Resource > Application
2. Choose **GitHub App** (requires connecting GitHub first)
3. Select the repo and branch
4. Set the **build pack**: Dockerfile (preferred), Nixpacks, or Static
5. Set the **domain**: Coolify auto-provisions an SSL cert via Let's Encrypt
6. Click **Deploy**

### Dockerfile is the right default

Always have a Dockerfile in the repo. It gives you full control over the build environment. Nixpacks is convenient for quick experiments but opaque for debugging.

## Environment Variables

Set env vars in Coolify, not in the repo.

- In the application settings: **Environment Variables** tab
- Add each key-value pair
- Mark secrets as **sensitive** (they are masked in logs)

### Secrets the app needs at runtime

Set them in Coolify. They are injected at deploy time.

### Secrets the Dockerfile needs at build time

Use Docker `--secret` mounts or ARG + ENV. Do not COPY secret files into the image.

```dockerfile
# Good: use a build arg, never a COPY
ARG API_KEY
ENV API_KEY=$API_KEY
```

## Database Services

Coolify can deploy Postgres, Redis, MySQL, and others as managed services.

1. Project > Environment > Add Resource > Database
2. Choose Postgres / Redis / etc.
3. Set a name and password
4. The connection string is automatically available to co-located apps

### Connecting your app to the database

Inside Coolify, services in the same project/environment can reach each other by container name:

```
DATABASE_URL=postgresql://user:password@MY_POSTGRES_CONTAINER_NAME/mydb
REDIS_URL=redis://MY_REDIS_CONTAINER_NAME:6379
```

Coolify injects these automatically if you link the resources.

## Domains and HTTPS

### Setting a custom domain

1. In the app settings, set the domain to `myapp.yourdomain.com`
2. In your DNS provider, add an A record: `myapp` pointing to your server IP
3. Coolify/Traefik auto-requests an SSL cert from Let's Encrypt
4. Allow a few minutes for cert provisioning

### Wildcard certs

For `*.yourdomain.com`, configure Traefik with a DNS-01 challenge instead of HTTP-01.
This requires API access to your DNS provider.

### Troubleshooting SSL

```bash
# Check Traefik logs
docker logs traefik --tail 100 2>&1 | grep -i 'cert\|acme\|error'

# Verify port 80 and 443 are open
ufw status
ss -tlnp | grep -E '80|443'
```

## Deploy Workflow

The standard deploy cycle:

```
1. Push to the tracked branch (usually main)
2. Coolify receives the GitHub webhook
3. Coolify pulls the new code and runs docker build
4. New container starts
5. Old container is replaced (zero-downtime if health check passes)
```

### Manual deploy

Trigger from the Coolify UI: **Redeploy** button on the application page.

Or via the API (used by the `coolify-deploy` skill):

```bash
curl -X POST \
  -H "Authorization: Bearer $COOLIFY_API_KEY" \
  "$COOLIFY_URL/api/v1/applications/$APP_UUID/start"
```

## Rollbacks

Coolify keeps previous build images until they are pruned.

To roll back:
1. App > Deployments tab
2. Find the previous successful deploy
3. Click **Rollback to this deployment**

If you need a true rollback in code:

```bash
git revert HEAD
git push
# Coolify auto-deploys the revert
```

## Deployment Checklist

Before deploying a new service to production:

- [ ] Dockerfile builds locally without errors
- [ ] All required env vars documented and set in Coolify
- [ ] Health check endpoint responds correctly
- [ ] Database migrations run successfully in staging (if applicable)
- [ ] Domain DNS record pointing to server
- [ ] UFW has port 80 and 443 open
- [ ] No secrets hardcoded in the image
- [ ] Container name noted for routing and log access
- [ ] Deployment logs reviewed: no ERROR lines

**Major version bumps:** require a test environment first. Never bump a major version directly to production.